# Model Train

In [1]:
import pandas as pd 
import numpy as np 

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, RobustScaler, FunctionTransformer, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer


In [2]:
df = pd.read_csv("/kaggle/input/insurance/insurance.csv")
df.head()


,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520


## Train/Validation/Test Split

In [3]:
X = df.drop('charges', axis=1)
y = df['charges']

y_binned = pd.qcut(y, q = 10, duplicates='drop')

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size = 0.2, stratify = y_binned,random_state = 42
)

y_temp_binned = pd.qcut(y_temp, q=10, duplicates='drop')

X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size = 0.25, stratify = y_temp_binned, random_state = 42 
)

print("Train:", X_train.shape, y_train.shape)
print("Validation:", X_val.shape, y_val.shape)
print("Test:", X_test.shape, y_test.shape)


Train: (802, 6) (802,)
Validation: (268, 6) (268,)
Test: (268, 6) (268,)


## Outlier Removal

In [4]:
from scipy.stats import zscore
import numpy as np
import pandas as pd

num_cols = X_train.select_dtypes(include=['int64', 'float64']).columns.tolist()
cat_cols = X_train.select_dtypes(include=['object']).columns.tolist()

print(f"Numeric columns: {num_cols}")
print(f"Categorical columns: {cat_cols}")

for col in num_cols:
    initial_skew = X_train[col].skew(skipna=True)
    print(f"\nColumn: {col}")
    print(f"  Initial skew: {initial_skew:.4f}")

    if abs(initial_skew) > 1:
        q1, q3 = X_train[col].quantile([0.25, 0.75])
        IQR = q3 - q1
        lower, upper = q1 - 1.5 * IQR, q3 + 1.5 * IQR
        mask = (X_train[col] >= lower) & (X_train[col] <= upper)
        outliers = (~mask).sum()
        X_train.loc[~mask, col] = np.nan
        print(f"  Used IQR — Outliers removed: {outliers}")
    else:
        z_scores = np.abs(zscore(X_train[col].dropna()))
        mask = (z_scores <= 3)
        outliers = (~mask).sum()

        valid_idx = X_train[col].dropna().index
        X_train.loc[valid_idx[~mask], col] = np.nan
        print(f"  Used Z-score — Outliers removed: {outliers}")

    new_skew = X_train[col].skew(skipna=True)
    print(f"  New skew: {new_skew:.4f}")


Numeric columns: ['age', 'bmi', 'children']
Categorical columns: ['sex', 'smoker', 'region']

Column: age
  Initial skew: 0.0597
  Used Z-score — Outliers removed: 0
  New skew: 0.0597

Column: bmi
  Initial skew: 0.2725
  Used Z-score — Outliers removed: 2
  New skew: 0.1874

Column: children
  Initial skew: 0.9449
  Used Z-score — Outliers removed: 11
  New skew: 0.7244
